# LangExtract — Quick Exploration
Testing LangExtract on German/English job postings using local Ollama (Llama 3.1 8B).  
No API key needed.

**Goal:** See how LangExtract extracts SKILL and TOOL entities, and compare feel vs. our fine-tuned model.

## 1. Install

In [ ]:
# Run once
!pip install langextract --break-system-packages -q

In [74]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env from project root

# LangExtract's Google Gemini provider reads GOOGLE_API_KEY
os.environ["GOOGLE_API_KEY"] = os.environ["LANGEXTRACT_API_KEY"]

print("API key loaded")

API key loaded


## 2. Imports + Verify Ollama is running

In [75]:
import requests
import langextract as lx
from langextract.providers.ollama import OllamaLanguageModel

OLLAMA_URL = "http://host.docker.internal:11434"

# Check Ollama is running
try:
    r = requests.get(OLLAMA_URL)
    print("Ollama is running")
except Exception:
    print("Ollama is NOT running — start it with: ollama serve")

ollama_model = OllamaLanguageModel(
    model_id="llama3.1:8b",
    model_url=OLLAMA_URL,
)

Ollama is running


## 3. Sample Job Postings
One English, one German — same mix as the dataset.

In [ ]:
english_job = """
Requirements:
3+ years of experience in machine learning and data science.
Strong proficiency in Python and experience with PyTorch or TensorFlow.
Experience with cloud platforms (AWS, GCP, or Azure).
Familiarity with MLOps practices including Docker, Kubernetes, and CI/CD pipelines.
Knowledge of NLP techniques and transformer-based models (e.g. BERT, GPT).
Experience with SQL and data engineering concepts.
Knowledge of statistical modeling and A/B testing.
"""

german_job = """
Anforderungen:
Mehrjährige Erfahrung in Machine Learning und Datenanalyse.
Sehr gute Kenntnisse in Python sowie Erfahrung mit PyTorch oder TensorFlow.
Erfahrung mit Cloud-Plattformen (AWS, Azure oder GCP).
Kenntnisse in MLOps und Deployment-Tools wie Docker und Kubernetes.
Erfahrung mit NLP und Transformer-Modellen.
Gute SQL-Kenntnisse und Verständnis von Datenpipelines.
Kenntnisse in statistischer Modellierung und A/B-Testing.
"""
description = """
Dein Skillset:\nAbschluss in Informatik, im Wirtschaftsingenieurswesen oder vergleichbar\nErfahrung in Automatisierung von ML-Pipelines, Plattformbetrieb und Orchestrierung mit Kubernetes/OpenShift sowie in der Optimierung von Monitoring- und Logging-Lösungen\nKenntnisse in Infrastructure-as-Code (Ansible, Terraform), CI/CD-Prozessen, Versionsverwaltung (Git) und GPU-Integration (z. B. NVIDIA) für ressourcenintensive KI-Workloads\nSicherer Umgang mit Tools wie MLflow, Kubeflow, Airflow sowie Grafana und Kibana für Monitoring und Logging; Erfahrung mit Containerisierung und Virtualisierungstechnologien\nDeutsch und Englisch sehr gut in Wort und Schrift
"""

print("Job postings ready.")

## 4. Define Prompt + Few-Shot Examples
This is the core of LangExtract — a natural language prompt + 2 examples.

In [ ]:
import textwrap

prompt = textwrap.dedent("""
    Extract SKILL and TOOL entities from job posting requirements sections.
    
    SKILL: A technical competency, methodology, or domain knowledge.
    Examples: machine learning, NLP, deep learning, statistical modeling, MLOps, data engineering, CI/CD
    
    TOOL: A specific technology, programming language, framework, platform, or library.
    Examples: Python, PyTorch, TensorFlow, Docker, AWS, SQL, Kubernetes, scikit-learn
    
    Rules:
    - Use EXACT text from the source. Do not translate or paraphrase.
    - Do not extract soft skills (teamwork, communication).
    - Do not extract job titles or degree names.
    - Works in both German and English.
""")

examples = [
    lx.data.ExampleData(
        text="Experience with Python and machine learning. Familiarity with Docker and AWS.",
        extractions=[
            lx.data.Extraction(extraction_class="TOOL", extraction_text="Python"),
            lx.data.Extraction(extraction_class="SKILL", extraction_text="machine learning"),
            lx.data.Extraction(extraction_class="TOOL", extraction_text="Docker"),
            lx.data.Extraction(extraction_class="TOOL", extraction_text="AWS"),
        ]
    ),
    lx.data.ExampleData(
        text="Kenntnisse in Deep Learning und Datenanalyse. Erfahrung mit PyTorch und Kubernetes.",
        extractions=[
            lx.data.Extraction(extraction_class="SKILL", extraction_text="Deep Learning"),
            lx.data.Extraction(extraction_class="SKILL", extraction_text="Datenanalyse"),
            lx.data.Extraction(extraction_class="TOOL", extraction_text="PyTorch"),
            lx.data.Extraction(extraction_class="TOOL", extraction_text="Kubernetes"),
        ]
    ),
]

print("Prompt and examples ready.")

## 5. Run Extraction — English Job

In [ ]:
print("Running LangExtract on English job posting...")
print("(This may take 30-60 seconds with Llama 3.1 8B)\n")

result_en = lx.extract(
    text_or_documents=english_job,
    prompt_description=prompt,
    examples=examples,
    model_id="llama3.1:8b",
    model_url="http://host.docker.internal:11434", 
    fence_output=False,
    use_schema_constraints=False,
)


print("=== ENGLISH JOB — Extracted Entities ===")
for extraction in result_en.extractions:
    print(f"  [{extraction.extraction_class}] '{extraction.extraction_text}'")

Running LangExtract on English job posting...
(This may take 30-60 seconds with Llama 3.1 8B)



LangExtract: Processing, current=461 chars, processed=0 chars:  [00:30]

=== ENGLISH JOB — Extracted Entities ===
  [SKILL] 'machine learning'
  [SKILL] 'data science'
  [TOOL] 'Python'
  [TOOL] 'PyTorch'
  [TOOL] 'TensorFlow'
  [TOOL] 'AWS'
  [TOOL] 'GCP'
  [TOOL] 'Azure'
  [TOOL] 'Docker'
  [TOOL] 'Kubernetes'
  [TOOL] 'CI/CD pipelines'
  [SKILL] 'NLP techniques'
  [SKILL] 'transformer-based models'
  [TOOL] 'BERT'
  [TOOL] 'GPT'
  [TOOL] 'SQL'
  [SKILL] 'data engineering'
  [SKILL] 'statistical modeling'
  [SKILL] 'A/B testing'
  [SKILL] 'MLOps practices'


## 6. Run Extraction — German Job

In [ ]:
print("Running LangExtract on German job posting...")
print("(This may take 30-60 seconds with Llama 3.1 8B)\n")

result_de = lx.extract(
    text_or_documents=german_job,
    prompt_description=prompt,
    examples=examples,
    model_id="llama3.1:8b",
    model_url="http://host.docker.internal:11434",
    fence_output=False,
    use_schema_constraints=False,
)

print("=== GERMAN JOB — Extracted Entities ===")
for extraction in result_de.extractions:
    print(f"  [{extraction.extraction_class}] '{extraction.extraction_text}'")

Running LangExtract on German job posting...
(This may take 30-60 seconds with Llama 3.1 8B)



LangExtract: Processing, current=431 chars, processed=0 chars:  [00:20]

=== GERMAN JOB — Extracted Entities ===
  [SKILL] 'Machine Learning'
  [SKILL] 'Datenanalyse'
  [SKILL] 'MLOps'
  [SKILL] 'statistische Modellierung'
  [SKILL] 'A/B-Testing'
  [TOOL] 'Python'
  [TOOL] 'PyTorch'
  [TOOL] 'TensorFlow'
  [TOOL] 'AWS'
  [TOOL] 'Azure'
  [TOOL] 'GCP'
  [TOOL] 'Docker'
  [TOOL] 'Kubernetes'
  [SKILL] 'NLP'
  [SKILL] 'Transformer-Modelle'
  [TOOL] 'SQL'


## 7. Run Extraction — Real Job Description (German)
A real job description from the dataset — more complex, mixed structure, longer text.

In [ ]:
print("Running LangExtract on real job description (German)...")
print("(This may take 30-60 seconds with Llama 3.1 8B)\n")

result_real = lx.extract(
    text_or_documents=description,
    prompt_description=prompt,
    examples=examples,
    model_id="llama3.1:8b",
    model_url="http://host.docker.internal:11434",
    fence_output=False,
    use_schema_constraints=False,
)

print("=== REAL JOB — Extracted Entities ===")
for extraction in result_real.extractions:
    print(f"  [{extraction.extraction_class}] '{extraction.extraction_text}'")

## 8. Compare Side by Side
Both jobs describe the same role — do the extractions match?

In [ ]:
en_skills = [e.extraction_text for e in result_en.extractions if e.extraction_class == "SKILL"]
en_tools  = [e.extraction_text for e in result_en.extractions if e.extraction_class == "TOOL"]
de_skills = [e.extraction_text for e in result_de.extractions if e.extraction_class == "SKILL"]
de_tools  = [e.extraction_text for e in result_de.extractions if e.extraction_class == "TOOL"]

print("ENGLISH")
print(f"  SKILLs: {en_skills}")
print(f"  TOOLs:  {en_tools}")
print()
print("GERMAN")
print(f"  SKILLs: {de_skills}")
print(f"  TOOLs:  {de_tools}")
print()
print(f"English total: {len(en_skills) + len(en_tools)} entities")
print(f"German total:  {len(de_skills) + len(de_tools)} entities")

## 9. Check Source Grounding
LangExtract's key feature — every extraction is mapped back to its exact position in the text.

In [ ]:
print("=== Source Grounding (English) ===")
print("Each extraction shows where in the text it was found.\n")

for e in result_en.extractions:
    has_grounding = hasattr(e, 'char_start') and e.char_start is not None
    if has_grounding:
        snippet = english_job[e.char_start:e.char_end]
        print(f"  [{e.extraction_class}] '{e.extraction_text}' @ chars {e.char_start}-{e.char_end}")
        print(f"    Source text: '...{snippet}...'")
    else:
        print(f"  [{e.extraction_class}] '{e.extraction_text}' (no grounding available with this model)")

## 10. Gemini Test

End-to-end run on a real job posting from the dataset using the same data as `scripts/09b_extract_requirements_all.py`:

1. Load API key from `.env`
2. Pick first job from `data/processed/jobs_with_requirements.json` that has a non-null `requirements_section`
3. Run LangExtract entity extraction with `gemini-2.5-flash`
4. Fall back to `description_clean` if `requirements_section` is None

In [76]:
import json
import os
import textwrap
from dotenv import load_dotenv

load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.environ["LANGEXTRACT_API_KEY"]
print("API key loaded")

# Load first job that has a non-null requirements_section
with open("../../data/processed/jobs_with_requirements.json", encoding="utf-8") as f:
    all_jobs = json.load(f)

job = next(
    (j for j in all_jobs if j.get("requirements_section")),
    None,
)

# Fall back to description_clean if no requirements_section available
if job is None:
    print("No jobs with requirements_section found — falling back to description_clean")
    job = next(j for j in all_jobs if j.get("description_clean"))
    inference_text = job["description_clean"]
    used_requirements_section = False
else:
    inference_text = job["requirements_section"]
    used_requirements_section = True

print(f"Job title:                 {job.get('title', 'N/A')}")
print(f"Company:                   {job.get('companyName', 'N/A')}")
print(f"Language:                  {job.get('language', 'N/A')}")
print(f"Used requirements_section: {used_requirements_section}")
print(f"Inference text length:     {len(inference_text)} chars")
print(f"\n--- Inference text preview ---")
print(inference_text[:500])

API key loaded
Job title:                 Data Scientist - (Logistics, Seamless Deliveries)
Company:                   Delivery Hero
Language:                  English
Used requirements_section: True
Inference text length:     922 chars

--- Inference text preview ---
You excel at modelling business problems using machine learning techniques, covering feature engineering, model selection and design, as well as evaluation and optimisation to deliver value to the business.
Expertise in building and training advanced machine learning models with strong knowledge of the Python data toolkit, ideally centring around PyTorch, xgboost, and lightgbm.
Through the fluent use of SQL queries, you are able to explore our data universe and gather data to support your analys


In [79]:
import langextract as lx

# LangExtract uses model IDs without the "models/" prefix
LANGEXTRACT_MODEL = "gemini-2.5-flash-lite"

print(f"Running LangExtract with {LANGEXTRACT_MODEL}...")
print(f"Input: {'requirements_section' if used_requirements_section else 'description_clean (fallback)'}")
print(f"Text length: {len(inference_text)} chars\n")

result_gemini = lx.extract(
    text_or_documents=inference_text,
    prompt_description=prompt,
    examples=examples,
    model_id=LANGEXTRACT_MODEL,
    fence_output=True,
    use_schema_constraints=True,
)

skills = [e for e in result_gemini.extractions if e.extraction_class == "SKILL"]
tools  = [e for e in result_gemini.extractions if e.extraction_class == "TOOL"]

print("=== Extracted Entities ===")
for e in result_gemini.extractions:
    print(f"  [{e.extraction_class}] {e.extraction_text}")

print(f"\n=== Summary ===")
print(f"  Model          : {LANGEXTRACT_MODEL}")
print(f"  Input field    : {'requirements_section' if used_requirements_section else 'description_clean'}")
print(f"  Total entities : {len(result_gemini.extractions)}")
print(f"  SKILLs         : {len(skills)}")
print(f"  TOOLs          : {len(tools)}")

Running LangExtract with gemini-2.5-flash-lite...
Input: requirements_section
Text length: 922 chars



LangExtract: model=gemini-2.5-flash-lite, current=922 chars, processed=0 chars:  [00:02]

=== Extracted Entities ===
  [SKILL] machine learning
  [SKILL] feature engineering
  [SKILL] model selection
  [SKILL] model design
  [SKILL] evaluation
  [SKILL] optimisation
  [TOOL] Python
  [TOOL] PyTorch
  [TOOL] xgboost
  [TOOL] lightgbm
  [TOOL] SQL
  [SKILL] data analysis
  [TOOL] Kubernetes
  [TOOL] Docker
  [TOOL] Kafka
  [TOOL] GCP
  [TOOL] BigQuery
  [SKILL] MLOps

=== Summary ===
  Model          : gemini-2.5-flash-lite
  Input field    : requirements_section
  Total entities : 18
  SKILLs         : 8
  TOOLs          : 10


## 11. Observations

Fill this in after running:

- Did LangExtract extract the expected entities?
- Did it handle German correctly?
- Were there false positives (wrong entities)?
- Were there false negatives (missed entities)?
- Did source grounding work?
- How does this compare to GLiNER (zero-shot, F1=0.27) in feel?

Key difference from the original Llama annotation:  
LangExtract structures the output AND grounds it to source positions automatically,  
whereas `08_annotate_llm.py` required manual span-mapping (the step that failed ~15% of the time).